# CityLearn Forecasting - Example Usage

This notebook demonstrates how to use the various forecasting models for time series prediction.

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from models import (
    LSTMForecaster,
    LSTMAutoencoder,
    LSTMAttention,
    LSTMEncoderDecoder,
    TimesNet
)
from data import TimeSeriesDataset
from utils import Trainer, calculate_metrics, plot_predictions

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Generate Synthetic Data

In [ ]:
# Generate synthetic time series data
np.random.seed(42)
t = np.linspace(0, 100, 10000)
data = np.sin(t) + 0.5 * np.sin(3 * t) + 0.3 * np.sin(5 * t) + np.random.normal(0, 0.1, len(t))
data = data.reshape(-1, 1)

# Plot the data
plt.figure(figsize=(15, 4))
plt.plot(data[:1000])
plt.title('Synthetic Time Series Data')
plt.xlabel('Time Steps')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: {data.shape}")

## 3. Create Dataset and DataLoaders

In [ ]:
# Create dataset
dataset = TimeSeriesDataset(
    data=data,
    seq_len=96,
    pred_len=24,
    stride=1,
    scale=True
)

# Split into train and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Input shape: {dataset[0][0].shape}")
print(f"Target shape: {dataset[0][1].shape}")

## 4. Train LSTM Model

In [ ]:
# Create LSTM model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

lstm_model = LSTMForecaster(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    output_size=24,
    dropout=0.2
)

print(f"Model parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

In [ ]:
# Setup training
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = Trainer(
    model=lstm_model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler,
    grad_clip=1.0
)

# Train the model
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=50,
    verbose=True,
    save_best=True,
    save_path='lstm_best.pth',
    early_stopping=10
)

## 5. Evaluate and Visualize Results

In [ ]:
# Make predictions
lstm_model.eval()
predictions = []
targets = []

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        pred = lstm_model(batch_x)
        predictions.append(pred.cpu().numpy())
        targets.append(batch_y.numpy())

predictions = np.concatenate(predictions, axis=0)
targets = np.concatenate(targets, axis=0)

# Calculate metrics
metrics = calculate_metrics(targets, predictions)
print("\nEvaluation Metrics:")
for name, value in metrics.items():
    print(f"{name.upper()}: {value:.6f}")

In [ ]:
# Plot predictions
plot_predictions(
    y_true=targets.flatten()[:500],
    y_pred=predictions.flatten()[:500],
    title='LSTM Predictions vs Ground Truth'
)

## 6. Compare Multiple Models

In [ ]:
# Train LSTM with Attention
attention_model = LSTMAttention(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    output_size=24,
    dropout=0.2
)

optimizer = torch.optim.Adam(attention_model.parameters(), lr=0.001)
trainer_attention = Trainer(
    model=attention_model,
    optimizer=optimizer,
    criterion=criterion,
    device=device
)

history_attention = trainer_attention.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
    verbose=True
)

In [ ]:
# Compare models
from utils import plot_multiple_forecasts

attention_predictions = trainer_attention.predict(val_loader)

plot_multiple_forecasts(
    y_true=targets.flatten()[:500],
    forecasts={
        'LSTM': predictions.flatten()[:500],
        'LSTM Attention': attention_predictions.flatten()[:500]
    },
    title='Model Comparison'
)

## 7. Using TimesNet

In [ ]:
# Create TimesNet model
timesnet_model = TimesNet(
    input_size=1,
    seq_len=96,
    pred_len=24,
    d_model=64,
    d_ff=128,
    num_layers=2,
    top_k=5,
    dropout=0.1
)

optimizer = torch.optim.Adam(timesnet_model.parameters(), lr=0.001)
trainer_timesnet = Trainer(
    model=timesnet_model,
    optimizer=optimizer,
    criterion=criterion,
    device=device
)

history_timesnet = trainer_timesnet.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
    verbose=True
)

print("\nTimesNet training completed!")

## 8. Save and Load Models

In [ ]:
# Save model
trainer.save_checkpoint('lstm_final.pth')
print("Model saved!")

# Load model
trainer.load_checkpoint('lstm_final.pth')
print("Model loaded!")